# Data Quality Analysis
Automated DQ validation framework.

The `DQEngine` runs a battery of checks – completeness, uniqueness,
freshness, volume, schema, value ranges, and email validity – against
each warehouse fact/dim table, then calculates a weighted DQ score.


In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path('.').parent))

import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import sqlite3

pd.set_option('display.max_columns', 30)
pd.set_option('display.max_colwidth', 80)

from datetime import datetime
from config import BASE_DIR, WAREHOUSE_DB, DQ_THRESHOLDS

print(f"Warehouse DB: {WAREHOUSE_DB}")
print(f"DQ Thresholds: {DQ_THRESHOLDS}")


In [ ]:
from src.extractors import CRMExtractor, MetaAdsExtractor, GoogleAdsExtractor, GA4Extractor, EmailPlatformExtractor
from src.transformers import (SchemaStandardizer, CRMTransformer, AdsTransformer,
                               WebTransformer, EmailTransformer, IdentityResolver)
from src.loaders import WarehouseLoader

END   = datetime(2025, 3, 1)
START = datetime(2025, 1, 1)

# Extract
crm_ext   = CRMExtractor({"accounts_count": 150, "contacts_count": 400,
                           "opportunities_count": 80, "activities_count": 600})
meta_ext  = MetaAdsExtractor({"campaigns": 4})
gads_ext  = GoogleAdsExtractor({"campaigns": 3})
ga4_ext   = GA4Extractor({"sessions_count": 1500, "events_count": 5000})
email_ext = EmailPlatformExtractor({"campaigns": 6, "sends_count": 1500})

crm_raw   = crm_ext.extract(START, END)
meta_raw  = meta_ext.extract(START, END)
gads_raw  = gads_ext.extract(START, END)
ga4_raw   = ga4_ext.extract(START, END)
email_raw = email_ext.extract(START, END)

# Transform
std = SchemaStandardizer()
ads_t  = AdsTransformer()
web_t  = WebTransformer()
email_t_obj = EmailTransformer()

contacts_std = std.transform(crm_ext.contacts_df.copy(), "crm")
unified_ads  = ads_t.transform(meta_data=meta_raw, google_data=gads_raw)
sessions     = ga4_raw[ga4_raw['record_type'] == 'session'].copy()                if 'record_type' in ga4_raw.columns else ga4_raw.copy()
web_sessions = web_t.transform(sessions)
email_eng    = email_t_obj.transform(email_raw)

# Load to warehouse
loader = WarehouseLoader()
loader.create_tables()

try:
    loader.load(crm_ext.accounts_df, "dim_accounts")
    loader.load(contacts_std, "dim_contacts")
    loader.load(unified_ads, "fact_ad_performance")
    loader.load(web_sessions, "fact_web_sessions")
    loader.load(email_eng, "fact_email_engagement")
    print("Warehouse loaded successfully.")
except Exception as e:
    print(f"Load note: {e}")


In [ ]:
from src.quality import DQEngine

# Read tables back from warehouse
conn = sqlite3.connect(WAREHOUSE_DB)
tables = {}
table_names = ['dim_accounts', 'dim_contacts', 'fact_ad_performance',
               'fact_web_sessions', 'fact_email_engagement']
for tname in table_names:
    try:
        tables[tname] = pd.read_sql(f"SELECT * FROM {tname} LIMIT 5000", conn)
        print(f"  {tname}: {len(tables[tname]):,} rows, {tables[tname].shape[1]} cols")
    except Exception as e:
        print(f"  {tname}: not available ({e})")
conn.close()

# Fall back to in-memory DataFrames if warehouse is empty
if not tables:
    print("\nUsing in-memory DataFrames for DQ checks.")
    tables = {
        "dim_accounts":        crm_ext.accounts_df,
        "dim_contacts":        contacts_std,
        "fact_ad_performance": unified_ads,
        "fact_web_sessions":   web_sessions,
        "fact_email_engagement": email_eng,
    }


In [ ]:
table_configs = {
    "dim_accounts": {
        "key_columns": ["account_id"],
        "date_column": "created_date",
        "expected_columns": ["account_id", "name", "industry", "region"],
    },
    "dim_contacts": {
        "key_columns": ["contact_id"],
        "date_column": "created_date",
        "has_email": True,
        "email_column": "email",
    },
    "fact_ad_performance": {
        "key_columns": ["campaign_id", "date"],
        "date_column": "date",
        "range_checks": [
            {"column": "spend",       "min": 0},
            {"column": "clicks",      "min": 0},
            {"column": "impressions", "min": 0},
        ],
    },
    "fact_web_sessions": {
        "date_column": "date",
    },
    "fact_email_engagement": {
        "date_column": "date",
    },
}

dq_engine = DQEngine()
print("Running DQ checks…")
dq_report = dq_engine.run_checks(tables, table_configs)

print()
print(f"DQ Score : {dq_report['dq_score']}/100")
print(f"Status   : {dq_report['status']}")
print(f"Checks   : {dq_report['checks_run']} run | "
      f"{dq_report['checks_passed']} passed | "
      f"{dq_report['checks_warning']} warning | "
      f"{dq_report['checks_failed']} failed")
print(f"Ran in   : {dq_report['execution_time_seconds']}s")


In [ ]:
# ── DQ check results table ────────────────────────────────────────────────
results_df = pd.DataFrame(dq_report['results'])
display_cols = [c for c in ['table', 'check_name', 'status', 'severity',
                              'metric_value', 'threshold', 'details']
                if c in results_df.columns]
print("=== DQ Check Results ===")
display(results_df[display_cols].sort_values(['status', 'severity'],
                                              ascending=[True, True]).reset_index(drop=True))


In [ ]:
# ── Visualise DQ check outcomes ───────────────────────────────────────────
status_counts = results_df['status'].value_counts()
colors_map    = {'PASS': '#4CAF50', 'WARNING': '#FF9800', 'FAIL': '#F44336'}
bar_colors    = [colors_map.get(s, '#9E9E9E') for s in status_counts.index]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle(f"DQ Summary – Score {dq_report['dq_score']}/100 | Status: {dq_report['status']}",
             fontsize=13, fontweight='bold')

# Left: status counts
bars = ax1.bar(status_counts.index, status_counts.values, color=bar_colors, edgecolor='white', width=0.5)
ax1.bar_label(bars, padding=3)
ax1.set_title("Check Outcomes")
ax1.set_ylabel("Count")

# Right: check result heatmap by table × check_name
if 'table' in results_df.columns and 'check_name' in results_df.columns:
    pivot = results_df.pivot_table(index='table', columns='check_name',
                                    values='status', aggfunc='first')
    status_num = pivot.replace({'PASS': 1, 'WARNING': 0.5, 'FAIL': 0, None: -1})
    im = ax2.imshow(status_num.values.astype(float), cmap='RdYlGn', vmin=0, vmax=1, aspect='auto')
    ax2.set_xticks(range(len(pivot.columns)))
    ax2.set_yticks(range(len(pivot.index)))
    ax2.set_xticklabels(pivot.columns, rotation=45, ha='right', fontsize=8)
    ax2.set_yticklabels(pivot.index, fontsize=8)
    ax2.set_title("Check Heatmap (green=PASS, red=FAIL)")
    plt.colorbar(im, ax=ax2, fraction=0.04, pad=0.04, label='Score')

plt.tight_layout()
plt.savefig(BASE_DIR / "visuals" / "nb04_dq_results.png", dpi=100, bbox_inches='tight')
plt.show()
print("Plot saved.")


In [ ]:
# ── Deep dive into any failed checks ─────────────────────────────────────
failed = results_df[results_df['status'] == 'FAIL']
warnings_df = results_df[results_df['status'] == 'WARNING']

if not failed.empty:
    print(f"=== FAILED Checks ({len(failed)}) ===")
    fail_cols = [c for c in ['table', 'check_name', 'severity', 'metric_value',
                               'threshold', 'affected_rows', 'details'] if c in failed.columns]
    display(failed[fail_cols].reset_index(drop=True))
else:
    print("No failed checks.")

if not warnings_df.empty:
    print(f"\n=== WARNING Checks ({len(warnings_df)}) ===")
    warn_cols = [c for c in ['table', 'check_name', 'severity', 'metric_value',
                               'threshold', 'details'] if c in warnings_df.columns]
    display(warnings_df[warn_cols].reset_index(drop=True))


In [ ]:
from src.quality import DQReporter

reporter = DQReporter()
try:
    report_text = reporter.generate_report(dq_report)
    print(report_text[:3000])
    if len(report_text) > 3000:
        print(f"\n... (truncated – {len(report_text)} chars total)")
except Exception as e:
    print(f"DQReporter error: {e}")
    print("\nRaw DQ report dict:")
    for k, v in dq_report.items():
        if k != 'results':
            print(f"  {k}: {v}")
